In [1]:
#-----------------------------------------------------Data Ingestion & Preprocessing----------------------------------------------------
from modules.loader import load_documents
from modules.splitter import split_text
from modules.vectorstore import create_vector_db
from modules.retriever import query_db

c:\Users\Anik HC\Desktop\Everything Coding\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Anik HC\Desktop\Everything Coding\python\capstone_project\modules\vectorstore.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12547.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.positi

In [2]:
DATA_PATH = "data/"
DB_PATH = "db/"

In [3]:
docs = load_documents(DATA_PATH)
print(f"Loaded {len(docs)} pages")

chunks = split_text(docs)
print(f"Created {len(chunks)} chunks")

db = create_vector_db(chunks, DB_PATH)
print("Database created!")

Loaded 762 pages
Created 2796 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11037.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database created!


c:\Users\Anik HC\Desktop\Everything Coding\python\capstone_project\modules\vectorstore.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [4]:
query = "What is normalization in database?"
results = query_db(query, k=5)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:300])


Result 1:
9. Introduction to Database Normalization 
 
Database normalization is the process of organizing the attributes of the 
database to reduce or eliminate  data redundancy (having the same data but 
at different places).  
Problems because of data redundancy: Data redundancy unnecessarily 
increases th

Result 2:
adhere to the principles of normalization. Normalization involves organizing 
data into tables and applying rules to ensure data is stored in a consistent and 
efficient manner. By reducing data redundancy and ensuring data integrity, 
normalization helps to eliminate anomalies and improve the overa

Result 3:
goals of Normalization include: 
 It helps in vacating all the repeated data from the database. 
 It helps in removing undesirable deletion, insertion, and update 
anomalies. 
 It helps in making a proper and useful relationship between tables.   
Advantages Anomalies in Relational Model 
 Data 

Result 4:
The features of database normalization are as follows

In [5]:
#------------------------------------------------LLM Integration (Response Generation)--------------------------------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

In [6]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [7]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using ONLY the provided context.

Rules:
- Give a SHORT and clear answer (3–5 lines)
- Do NOT copy the context directly
- Summarize in your own words
- Use bullet points if helpful
- If answer is not in context, say:
  "I don't know based on the provided documents"

Context:
{context}

Question:
{question}

Answer:
""")

In [8]:
def generate_answer(query):
    # 1. Retrieve docs
    docs = query_db(query, k=5)

    # 2. Prepare context
    context = "\n\n".join(
        [doc.page_content[:500] for doc in docs]
    )

    # 3. Create prompt
    final_prompt = prompt.format(
        context=context,
        question=query
    )

    # 4. Get response
    response = llm.invoke(final_prompt)

    # Clean output
    answer = response.content.strip()

    # Optional: limit length
    if len(answer) > 800:
        answer = answer[:800] + "..."

    return answer

In [9]:
query = "What is normalization in DBMS?"
answer = generate_answer(query)

print(answer)

Normalization in DBMS refers to a process aimed at organizing data attributes to minimize or eliminate redundancy, thereby reducing inconsistencies and improving overall database quality. It involves structuring data into tables and applying rules to ensure consistent and efficient storage. The main goals of normalization are to remove repeated data, eliminate anomalies during insertion, deletion, and update operations, and establish proper relationships between tables. Normalization also helps enforce data integrity through various constraints such as primary keys and foreign keys.


In [10]:
generate_answer("What is normalization?")

'Normalization is a process used in databases to organize attributes and eliminate data redundancy, aiming to reduce or eliminate three anomalies (Insert, Update, and Delete) caused by redundant data. It helps maintain data integrity through various constraints like primary keys and foreign keys, standardizes the data, and ensures proper relationships between tables.'

In [11]:
generate_answer("Who is Virat Kohli?")

"I don't know based on the provided documents. The context does not contain any information about Virat Kohli."

In [12]:
#-----------------------------------------------------Conversational Memory System----------------------------------------------------
chat_history = []

def generate_answer_with_memory(query):
    global chat_history

    # 1. Rewrite vague follow-up into standalone question
    final_query = rewrite_query_with_history(query, chat_history)

    # 2. Retrieve docs using rewritten query
    docs = query_db(query, k=5)

    # 3. Build context
    context = "\n\n".join([doc.page_content[:500] for doc in docs])

    # 4. Include recent conversation
    history_text = "\n".join(chat_history[-6:])

    answer_prompt = f"""
You are a helpful assistant.

Use the provided context and conversation history to answer the question.

Rules:
- Answer clearly and briefly
- Prefer the current topic from conversation history when the user asks a follow-up
- Do not introduce unrelated topics
- If the answer is not in the context, say:
  "I don't know based on the provided documents"

Conversation history:
{history_text}

Context:
{context}

Question:
{final_query}

Answer:
"""

    response = llm.invoke(answer_prompt)
    answer = response.content.strip()

    # 5. Save original user query and answer
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {answer}")

    return answer

In [13]:
follow_up_starts = [
    "why", "how", "explain", "give", "what about", "is it", "does it",
    "can it", "tell me more", "advantages", "disadvantages"
]

def is_follow_up(query: str) -> bool:
    q = query.strip().lower()
    return any(q.startswith(x) for x in follow_up_starts)

In [14]:
def rewrite_query_with_history(query, history):
    if not history or not is_follow_up(query):
        return query

    recent_history = "\n".join(history[-4:])

    rewrite_prompt = f"""
You are rewriting a user's follow-up question into a fully standalone question.

Conversation history:
{recent_history}

Follow-up question:
{query}

Rewrite it into one clear standalone question.
Only return the rewritten question.
"""
    rewritten = llm.invoke(rewrite_prompt).content.strip()
    return rewritten

In [15]:
print(generate_answer_with_memory("What is normalization?"))
print(generate_answer_with_memory("Why is it important?"))

Normalization is a process to eliminate data redundancy in a database and organize attributes to reduce or eliminate data anomalies such as insert and update anomalies. It helps in maintaining data integrity, standardizing data, and improving the overall quality of the database.
Normalization in a database helps to eliminate unnecessary duplication of data, reduce data redundancy, and organize attributes to minimize data anomalies such as insert and update anomalies. It improves data integrity, simplifies database management, avoids update anomalies, and enhances the overall design of the database for better flexibility and adaptability to changing business needs.


In [16]:
#-------------------------------------------------LangGraph Workflow Implementation-----------------------------------------------------
from langgraph.graph import StateGraph
from typing import TypedDict, List

In [17]:
class State(TypedDict):
    question: str
    context: str
    answer: str
    history: List[str]
    route: str
    evaluation: str

In [18]:
def memory_node(state: State):
    history = state.get("history", [])
    history.append(f"User: {state['question']}")
    return {"history": history}

In [19]:
def retrieve_node(state: State):
    query = state["question"]
    history = state.get("history", [])

    # Detect follow-up
    follow_up_words = ["why", "how", "explain", "advantages", "disadvantages"]
    is_follow_up = any(query.lower().startswith(w) for w in follow_up_words)

    # Rewrite query if needed
    if history and is_follow_up:
        recent_history = "\n".join(history[-4:])

        rewrite_prompt = f"""
Rewrite the following follow-up question into a standalone question.

Conversation:
{recent_history}

Follow-up question:
{query}

Only return the rewritten question.
"""

        query = llm.invoke(rewrite_prompt).content.strip()

    # Retrieve documents using rewritten query
    docs = query_db(query, k=5)

    context = "\n\n".join(
        [doc.page_content[:500] for doc in docs]
    )

    return {"context": context}

In [20]:
def answer_node(state: State):
    history_text = "\n".join(state.get("history", [])[-6:])

    prompt_text = f"""
You are a helpful assistant.

Use the provided context and conversation history.

Rules:
- Keep answers short and clear
- Do not introduce unrelated topics
- If answer not found, say:
  "I don't know based on the provided documents"

Conversation:
{history_text}

Context:
{state['context']}

Question:
{state['question']}

Answer:
"""

    response = llm.invoke(prompt_text)
    answer = response.content.strip()

    # update history
    history = state.get("history", [])
    history.append(f"Assistant: {answer}")

    return {"answer": answer, "history": history}

In [21]:
builder = StateGraph(State)

builder.add_node("memory", memory_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("answer", answer_node)

builder.set_entry_point("memory")
builder.add_edge("memory", "retrieve")
builder.add_edge("retrieve", "answer")

graph = builder.compile()

In [22]:
state = {
    "question": "What is normalization?",
    "history": []
}

result = graph.invoke(state)

print(result["answer"])

Normalization is a process to eliminate data redundancy in databases and adhere to principles that help reduce three anomalies (Insert, Update, and Delete) caused by data redundancy. It helps organize the attributes of the database into tables, standardize data, and ensure proper relationships between tables for improved data integrity and overall quality of the database.


In [23]:
state = graph.invoke({
    "question": "What is normalization?",
    "history": []
})
print(state["answer"])
state = graph.invoke({
    "question": "Why is it important?",
    "history": state["history"]
})

print(state["answer"])

Normalization is a process in databases that organizes attributes to reduce or eliminate data redundancy and anomalies. It helps in maintaining data integrity by enforcing constraints such as primary keys, foreign keys, and referential integrity. The goal is to ensure each table contains only one type of data and relationships between tables are clearly defined, which helps avoid insert, update, and delete anomalies. Normalization also standardizes the data in the database for consistent storage.
Normalization is important because it helps in reducing data redundancy, eliminating data anomalies, improving database design, standardizing data, making the database more compact, and ensuring data integrity through various constraints such as primary keys, foreign keys, and referential integrity. It also makes the database easier to maintain and change as business needs evolve.


In [24]:
#-----------------------------------------------------------Adding Router--------------------------------------------------------------
def router_node(state: State):
    question = state["question"]

    routing_prompt = f"""
Classify the question into one of two categories:

1. "retrieval" → if it is about database, normalization, DBMS topics
2. "tool" → if it is math, calculation, or general knowledge

Question:
{question}

Answer ONLY one word: retrieval or tool
"""

    decision = llm.invoke(routing_prompt).content.strip().lower()

    if "tool" in decision:
        return {"route": "tool"}
    else:
        return {"route": "retrieve"}

In [25]:
def tool_node(state: State):
    question = state["question"]

    tool_prompt = f"""
Solve the following question:

{question}

Give only the final answer.
"""

    response = llm.invoke(tool_prompt)

    return {"context": response.content.strip()}

In [27]:
#------------------------------------------------------------Evaluation-----------------------------------------------------------------
def evaluation_node(state: State):
    question = state["question"]
    answer = state["answer"]
    context = state.get("context", "")

    eval_prompt = f"""
You are evaluating an AI answer.

Question:
{question}

Answer:
{answer}

Context:
{context}

Evaluate:
1. Is the answer relevant to the question? (yes/no)
2. Is the answer supported by the context? (yes/no)

Respond in this format:
relevant: yes/no
faithful: yes/no
"""

    result = llm.invoke(eval_prompt).content.strip()

    return {"evaluation": result}

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(State)

builder.add_node("memory", memory_node)
builder.add_node("router", router_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("tool", tool_node)
builder.add_node("answer", answer_node)
builder.add_node("evaluation", evaluation_node)

builder.set_entry_point("memory")

builder.add_edge("memory", "router")

def route_decision(state: State):
    return state["route"]

builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "retrieve": "retrieve",
        "tool": "tool"
    }
)

builder.add_edge("retrieve", "answer")
builder.add_edge("tool", "answer")
builder.add_edge("answer", "evaluation")

graph = builder.compile()

In [29]:
state = graph.invoke({
    "question": "What is normalization?",
    "history": []
})

print(state["answer"])

Normalization is a process to eliminate data redundancy in databases, reducing or eliminating three anomalies (Insert, Update, and Delete) that can cause inconsistencies. It helps in organizing the attributes of the database to make each table contain only one type of data and clearly define relationships between tables. Normalization also standardizes data in the database by ensuring it is stored consistently and efficiently, adhering to principles set by E.F. Codd, the inventor of the Relational Database.


In [30]:
state = graph.invoke({
    "question": "Why is it important?",
    "history": state["history"]
})

print(state["answer"])

Normalization is important because it ensures data consistency, reduces redundancy, and improves database efficiency by organizing data logically and defining relationships between tables. This helps prevent inconsistencies caused by the three anomalies (Insert, Update, and Delete) and makes data easier to manage and maintain over time.


In [31]:
state = graph.invoke({
    "question": "What is 25 * 12?",
    "history": []
})

print(state["answer"])

300


In [32]:
state = graph.invoke({
    "question": "What is normalization?",
    "history": []
})

print(state["answer"])
print(state["evaluation"])

Normalization is a process to eliminate data redundancy in databases, reducing or eliminating three anomalies (Insert, Update, and Delete) that can cause inconsistencies. It helps in organizing the attributes of the database to make each table contain only one type of data and clearly define relationships between tables. Normalization also standardizes data in the database by ensuring it is stored consistently and efficiently, adhering to principles set by E.F. Codd, the inventor of the Relational Database.
relevant: yes
faithful: yes


In [33]:
state = graph.invoke({
    "question": "Why is it important?",
    "history": state["history"]
})

print(state["answer"])
print(state["evaluation"])

Normalization is important because it ensures data consistency, reduces redundancy, and improves database efficiency by organizing data logically and defining relationships between tables. This helps prevent inconsistencies caused by the three anomalies (Insert, Update, and Delete) and makes data easier to manage and maintain over time.
relevant: yes
faithful: yes


In [34]:
state = graph.invoke({
    "question": "What is 45 * 12?",
    "history": []
})

print(state["answer"])

540


In [35]:
test_cases = [
    "What is normalization?",
    "Why is normalization important?",
    "What is indexing?",
    "Explain database schema",
    "What is primary key?",
    "What is foreign key?",
    "Explain 3NF",
    "Advantages of normalization",
    "What is 25 * 12?",   # tool
    "Who is Virat Kohli?" # out-of-scope
]

In [36]:
for q in test_cases:
    state = graph.invoke({
        "question": q,
        "history": []
    })
    
    print("\nQuestion:", q)
    print("Answer:", state["answer"])
    print("Evaluation:", state["evaluation"])


Question: What is normalization?
Answer: Normalization is a process to eliminate data redundancy in databases and organize attributes to reduce or eliminate data anomalies such as insert, update, and deletion anomalies. It helps in maintaining data integrity by enforcing various constraints like primary keys, foreign keys, and referential integrity. Additionally, normalization standardizes the data in the database, ensuring it is stored consistently and efficiently.
Evaluation: relevant: yes
faithful: yes

Question: Why is normalization important?
Answer: Normalization is important because it helps in reducing data redundancy, avoiding data inconsistencies during insert, delete, and update operations, vacating repeated data from the database, removing undesirable deletion, insertion, and update anomalies, making a proper and useful relationship between tables, standardizing data, improving database design, and making it easier to maintain and change as the needs change.
Evaluation: re